# CashCompass: Tabular Model Training (one notebook for your CSV files)

Put whichever of these CSV files you have in one folder. The notebook **finds them, detects the columns, and trains the matching model**. Files you do not have are skipped with a clear message.

| CSV file | Model trained here | Output |
|---|---|---|
| `transactions.csv` | **Expense forecaster** (next-month expenses, P10/P50/P90) + **merchant to category classifier** | forecast range, category |
| `monthly_financials.csv` | **Savings-capacity forecaster** (next-month surplus, P10/P50/P90) | surplus range |
| `risk_questionnaire.csv` | **Risk profiler** (risk category + risk score from answers) | category, score |
| `financial_goals.csv` | **Goal planner** (required monthly saving, on-track status) | rules, not trained (no labels in the file) |
| `investment_catalogue.csv` | **Recommendation engine** (eligibility filter + suitability score) | ranked options |
| `recommendation_feedback.csv` | **Acceptance model** (will the user accept this suggestion?) | probability |

**Honest rules this notebook follows**
- Fewer than 3 months of data is treated as too short, and a model is only trained when there is enough data to test it properly.
- Every model is compared with a simple baseline. If the model does not beat it, the notebook says so.
- If `monthly_financials.csv` is missing, monthly totals are **derived from `transactions.csv`**.
- Nothing is sent anywhere. Everything runs on your machine.

## 0. Setup

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib joblib

In [ ]:
import os, re, json, hashlib, warnings, platform, datetime
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (HistGradientBoostingRegressor, HistGradientBoostingClassifier,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error, r2_score, roc_auc_score,
                             confusion_matrix)
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings("ignore")

SEED = 42
DATA_DIR = r"C:\Users\lenovo\Downloads"
SEARCH_DIRS = [DATA_DIR, "data/raw", "."]
FILE_MAP = {
    "transactions": "transactions.csv",
    "monthly_financials": "monthly_financials.csv",
    "risk_questionnaire": "risk_questionnaire.csv",
    "financial_goals": "financial_goals.csv",
    "investment_catalogue": "investment_catalogue.csv",
    "recommendation_feedback": "recommendation_feedback.csv",
}
COLUMN_OVERRIDE = {}
OUT_DIR = "outputs_tabular"
AS_OF_DATE = None
MIN_MONTHS = 3
MIN_FORECAST_ROWS = 16
np.random.seed(SEED)
os.makedirs(f"{OUT_DIR}/models", exist_ok=True)
RESULTS = {"seed": SEED, "sklearn": sklearn.__version__, "python": platform.python_version()}
STATUS = {}
BUNDLES = {}
print("pandas", pd.__version__, "| sklearn", sklearn.__version__)

## 1. Helpers: find files, detect columns, parse numbers and dates

If a column is detected wrongly, fix it with `COLUMN_OVERRIDE` in the setup cell, for example:
`COLUMN_OVERRIDE = {"transactions": {"date": "Txn Date", "amount": "Amt"}}`

In [ ]:
SYN = {
    "date": ["date", "transaction date", "txn date", "timestamp", "time"],
    "month": ["month", "period", "year month", "date"],
    "amount": ["amount", "amt", "value", "transaction amount"],
    "category": ["category", "spend category"],
    "merchant": ["merchant", "payee", "vendor", "description", "narration", "details"],
    "type": ["transaction type", "txn type", "type", "dr cr", "debit credit"],
    "user": ["user id", "userid", "user", "customer id", "customer", "client", "account id", "person"],
    "debit": ["debit amount", "debit", "withdrawal amount", "withdrawals", "withdrawal", "withdraw", "dr amount"],
    "credit": ["credit amount", "credit", "deposit amount", "deposits", "deposit", "cr amount"],
    "income": ["income", "salary", "earnings", "inflow"],
    "expenses": ["expenses", "expense", "spending", "spend", "outflow"],
    "emi": ["emi", "loan payment", "loan emi", "installment", "instalment"],
    "surplus": ["surplus", "savings", "net savings", "net cash flow", "saving"],
    "score": ["risk score", "score", "total score"],
    "risk_category": ["risk category", "risk level", "risk profile", "risk class", "category", "profile", "class"],
    "answers": ["answers", "answer", "responses", "response"],
    "goal": ["goal", "goal name", "name", "objective"],
    "target": ["target amount", "target", "goal amount", "amount needed"],
    "current": ["current savings", "current", "saved", "saved so far", "current amount"],
    "deadline": ["deadline", "target date", "due date", "end date"],
    "option": ["suggested option", "option", "suggested", "recommendation", "product", "investment", "instrument"],
    "label": ["accepted rejected", "accepted", "accept", "outcome", "decision", "response", "status", "rejected"],
    "reason": ["reason", "feedback", "comment", "remarks"],
    "inv_type": ["investment type", "type", "instrument", "name", "product"],
    "risk": ["risk", "risk level", "risk rating"],
    "liquidity": ["liquidity", "liquidity level"],
    "fees": ["fees", "fee", "expense ratio", "charges", "cost"],
    "lock_in": ["lock in", "lockin", "lock in period", "lock in months", "tenure", "maturity"],
    "eligibility": ["eligibility", "eligible", "criteria"],
}

def norm_name(name):
    return re.sub(r"[^a-z0-9]+", " ", str(name).lower()).strip()

def detect(columns, keys, exclude=()):
    names = {c: norm_name(c) for c in columns}
    for key in keys:
        for c, n in names.items():
            if n == key and not any(x in n for x in exclude):
                return c
    for key in keys:
        for c, n in names.items():
            if key in n and not any(x in n for x in exclude):
                return c
    return None

def pick(file_key, field, columns, exclude=()):
    forced = COLUMN_OVERRIDE.get(file_key, {}).get(field)
    if forced:
        if forced not in columns:
            raise KeyError(f"COLUMN_OVERRIDE[{file_key!r}][{field!r}]={forced!r} is not a column of that file: {list(columns)}")
        return forced
    return detect(columns, SYN[field], exclude)

def to_num(series):
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(float)
    cleaned = series.astype(str).str.replace(r"[^\d.\-]", "", regex=True).replace({"": np.nan, "-": np.nan, ".": np.nan})
    return pd.to_numeric(cleaned, errors="coerce")

def to_date(series):
    first = pd.to_datetime(series, errors="coerce")
    if first.isna().mean() > 0.3:
        second = pd.to_datetime(series, errors="coerce", dayfirst=True)
        if second.isna().mean() < first.isna().mean():
            return second
    return first

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def find_file(key):
    tokens = {"transactions": ["transaction"], "monthly_financials": ["monthly_fin", "monthly"], "risk_questionnaire": ["risk"],
              "financial_goals": ["goal"], "investment_catalogue": ["catalog"], "recommendation_feedback": ["feedback", "recommend"]}[key]
    for folder in SEARCH_DIRS:
        folder = Path(folder)
        exact = folder / FILE_MAP[key]
        if exact.is_file():
            return exact
    for folder in SEARCH_DIRS:
        folder = Path(folder)
        if folder.is_dir():
            for path in sorted(folder.glob("*.csv")):
                if any(t in path.name.lower() for t in tokens) and path.name not in FILE_MAP.values():
                    return path
    return None

def money(x):
    return f"{x:,.0f}"

def skip(key, why):
    STATUS[key] = f"SKIPPED: {why}"
    print(f"[{key}] SKIPPED: {why}")

## 2. Discover and load your files

In [ ]:
RAW = {}
for key in FILE_MAP:
    path = find_file(key)
    if path is None:
        STATUS[key] = "NOT FOUND"
        print(f"{key:26s} not found")
        continue
    frame = pd.read_csv(path, low_memory=False)
    frame.columns = [str(c).strip() for c in frame.columns]
    RAW[key] = frame
    print(f"{key:26s} {str(path)}  rows={len(frame)}  cols={list(frame.columns)}")
if not RAW:
    raise FileNotFoundError("No CSV files found. Set DATA_DIR in the setup cell.")

## 3. Transactions -> monthly totals

Each transaction row is classified as **income / expense / EMI**. EMI rows are found by keywords (`emi, loan, mortgage, instalment, repayment`) in the category or merchant text.

In [ ]:
INCOME_TYPE_RE  = r"credit|income|deposit|inflow|received|\bcr\b"
EXPENSE_TYPE_RE = r"debit|expense|withdraw|outflow|payment|purchase|\bdr\b"
INCOME_CAT_RE   = r"salary|income|bonus|interest|dividend|payroll|refund|cashback"
EMI_RE          = r"\bemi\b|loan|mortgage|instal+ment|repayment"

def tidy_transactions(raw):
    cols = list(raw.columns)
    c = {f: pick("transactions", f, cols, ("account type",) if f == "type" else ()) for f in ("date", "amount", "category", "merchant", "type", "user")}
    c_debit, c_credit = pick("transactions", "debit", cols), pick("transactions", "credit", cols)
    separate = bool(c_debit and c_credit and c_debit != c_credit
                    and to_num(raw[c_debit]).notna().mean() > 0.05 and to_num(raw[c_credit]).notna().mean() > 0.05)
    print("DETECTED transactions ->", c, "| separate debit/credit columns:", (c_debit, c_credit) if separate else None)
    if c["date"] is None or (c["amount"] is None and not separate):
        raise ValueError("transactions.csv needs a date column and an amount (or debit/credit) column (set COLUMN_OVERRIDE).")
    out = pd.DataFrame({"date": to_date(raw[c["date"]])})
    out["user"] = raw[c["user"]].astype(str) if c["user"] else "user_0"
    out["category"] = raw[c["category"]].astype(str).str.strip() if c["category"] else ""
    out["merchant"] = raw[c["merchant"]].astype(str).str.strip() if c["merchant"] else ""
    text = (out["category"] + " " + out["merchant"]).str.lower()
    if separate:
        credit, debit = to_num(raw[c_credit]).fillna(0), to_num(raw[c_debit]).fillna(0)
        amount = credit.where(credit > 0, debit)
        kind = pd.Series(np.where(credit > 0, "income", "expense"), index=raw.index, dtype=object)
    else:
        amount = to_num(raw[c["amount"]])
        kind = pd.Series(None, index=raw.index, dtype=object)
        if c["type"]:
            tv = raw[c["type"]].astype(str).str.lower()
            kind = kind.mask(tv.str.contains(INCOME_TYPE_RE, regex=True), "income")
            kind = kind.mask(tv.str.contains(EXPENSE_TYPE_RE, regex=True) & kind.isna(), "expense")
        has_negative = (amount < 0).any()
        income_by_category = out["category"].str.lower().str.contains(INCOME_CAT_RE, regex=True)
        fallback = np.where(income_by_category, "income", np.where(amount < 0, "expense", np.where(has_negative, "income", "expense")))
        kind = kind.where(kind.notna(), pd.Series(fallback, index=raw.index))
    kind = kind.where(~((kind == "expense") & text.str.contains(EMI_RE, regex=True)), "emi")
    out["kind"], out["amount"] = kind.values, amount.abs().values
    out = out.dropna(subset=["date", "amount"])
    return out[out["amount"] > 0].reset_index(drop=True)

def monthly_from_tidy(tidy, trim_partial=True):
    work = tidy.copy()
    work["month"] = work["date"].dt.to_period("M")
    pivot = work.pivot_table(index=["user", "month"], columns="kind", values="amount", aggfunc="sum", fill_value=0).reset_index()
    for k in ("income", "expense", "emi"):
        if k not in pivot:
            pivot[k] = 0.0
    pivot = pivot.rename(columns={"expense": "expenses", "user": "group"})
    span = work.groupby("user")["date"].agg(["min", "max"])
    parts = []
    for g, part in pivot.groupby("group"):
        part = part.sort_values("month").reset_index(drop=True)
        if trim_partial and len(part) > 2:
            lo, hi = span.loc[g, "min"], span.loc[g, "max"]
            if lo.day > 5:
                part = part.iloc[1:]
            if hi.day < 25:
                part = part.iloc[:-1]
        parts.append(part)
    monthly = pd.concat(parts, ignore_index=True)
    monthly["surplus"] = monthly["income"] - monthly["expenses"] - monthly["emi"]
    return monthly[["group", "month", "income", "expenses", "emi", "surplus"]]

TIDY = None
MONTHLY_FROM_TX = None
if "transactions" in RAW:
    TIDY = tidy_transactions(RAW["transactions"])
    print("\nrows:", len(TIDY), "| users:", TIDY["user"].nunique(), "| dates:", TIDY["date"].min().date(), "->", TIDY["date"].max().date())
    display(TIDY.groupby("kind")["amount"].agg(["count", "sum"]).round(0))
    if "income" not in set(TIDY["kind"]):
        print("WARNING: no income rows detected. Surplus from transactions will be wrong; check COLUMN_OVERRIDE['transactions']['type'].")
    MONTHLY_FROM_TX = monthly_from_tidy(TIDY)
    months_per_group = MONTHLY_FROM_TX.groupby("group")["month"].nunique()
    print("months per user (min/median/max):", months_per_group.min(), months_per_group.median(), months_per_group.max())
    if months_per_group.max() < MIN_MONTHS:
        print("TOO SHORT: fewer than 3 months of statements. The system must say it does not know.")
    display(MONTHLY_FROM_TX.head())

## 4. Monthly financials table (from `monthly_financials.csv`, else from transactions)

The notebook also checks **how your `surplus` column is calculated** (income - expenses - EMI, or income - expenses) so the numbers are interpreted correctly.

In [ ]:
def load_monthly_financials(raw):
    cols = list(raw.columns)
    c = {f: pick("monthly_financials", f, cols) for f in ("month", "income", "expenses", "emi", "surplus", "user")}
    print("DETECTED monthly_financials ->", c)
    if c["month"] is None or c["income"] is None or c["expenses"] is None:
        raise ValueError("monthly_financials.csv needs month, income and expenses columns (set COLUMN_OVERRIDE).")
    frame = pd.DataFrame({"group": raw[c["user"]].astype(str) if c["user"] else "user_0",
                          "month": to_date(raw[c["month"]]).dt.to_period("M"),
                          "income": to_num(raw[c["income"]]), "expenses": to_num(raw[c["expenses"]]),
                          "emi": to_num(raw[c["emi"]]).fillna(0) if c["emi"] else 0.0})
    provided = to_num(raw[c["surplus"]]) if c["surplus"] else None
    frame = frame.dropna(subset=["month", "income", "expenses"])
    if provided is not None:
        frame["surplus"] = provided.loc[frame.index]
        tol = max(0.02 * float(frame["income"].median()), 1.0)
        e_with = float((frame.income - frame.expenses - frame.emi - frame.surplus).abs().median())
        e_without = float((frame.income - frame.expenses - frame.surplus).abs().median())
        if e_with <= tol:
            print("surplus column = income - expenses - EMI  (OK)")
        elif e_without <= tol:
            print("surplus column = income - expenses  (expenses already include EMI)")
        else:
            print("WARNING: surplus does not match either formula; the provided surplus is used as the target.")
        frame["surplus"] = frame["surplus"].fillna(frame.income - frame.expenses - frame.emi)
    else:
        frame["surplus"] = frame.income - frame.expenses - frame.emi
    return frame.groupby(["group", "month"], as_index=False).sum(numeric_only=True)

MONTHLY, MONTHLY_SOURCE = None, None
if "monthly_financials" in RAW:
    MONTHLY, MONTHLY_SOURCE = load_monthly_financials(RAW["monthly_financials"]), "monthly_financials.csv"
elif MONTHLY_FROM_TX is not None:
    MONTHLY, MONTHLY_SOURCE = MONTHLY_FROM_TX, "derived from transactions.csv"
if MONTHLY is None:
    print("No monthly data available: forecasting sections will be skipped.")
else:
    counts = MONTHLY.groupby("group")["month"].nunique()
    print(f"monthly table source: {MONTHLY_SOURCE} | groups: {len(counts)} | months per group min/median/max: {counts.min()}/{counts.median()}/{counts.max()}")
    if counts.max() < MIN_MONTHS:
        print("TOO SHORT: fewer than 3 months. Forecasts would be meaningless.")
    display(MONTHLY.head())

## 5. Forecasting model (used for expenses and for surplus / savings capacity)

**Idea:** look at the last 2-3 months and predict next month, as a **range** (P10 / P50 / P90).

- Every money value is divided by the recent average income, so many people can be pooled.
- Candidates: simple baseline (average of recent months), Ridge, and gradient boosting (only with lots of rows). The best on a validation set wins.
- The range comes from the model's past errors on validation data (so it is honest about uncertainty).
- Split is by **time** (oldest to train, newest to test), never random.

In [ ]:
def make_lag_frame(monthly, target, lags):
    extras = [c for c in ("income", "expenses", "emi") if c != target and c in monthly.columns]
    rows = []
    for g, part in monthly.groupby("group"):
        part = part.sort_values("month").drop_duplicates("month").set_index("month")
        part = part.reindex(pd.period_range(part.index.min(), part.index.max(), freq="M"))
        for t in range(lags, len(part)):
            window = part.iloc[t - lags:t]
            goal = part.iloc[t][target]
            if pd.isna(goal) or window[[target] + extras].isna().any().any():
                continue
            use_income = "income" in window and window["income"].mean() > 0
            scale = max(float(window["income"].mean()) if use_income else float(window[target].abs().mean()), 1.0)
            feats = [window[target].iloc[-k] / scale for k in range(1, lags + 1)]
            for e in extras:
                feats += [window[e].iloc[-k] / scale for k in range(1, lags + 1)]
            month = part.index[t].month
            feats += [window[target].mean() / scale, np.sin(2 * np.pi * month / 12), np.cos(2 * np.pi * month / 12)]
            rows.append({"group": g, "month": part.index[t], "scale": scale, "y": goal / scale, "x": feats})
    return extras, pd.DataFrame(rows)

def split_by_time(frame):
    n = len(frame)
    n_test = max(4, round(0.20 * n))
    n_val = max(4, round(0.15 * n))
    n_train = n - n_test - n_val
    if n_train < 8:
        return None
    order = frame.sort_values("month", kind="stable").reset_index(drop=True)
    return order.iloc[:n_train], order.iloc[n_train:n_train + n_val], order.iloc[n_train + n_val:]

def stack(frame):
    return np.vstack(frame["x"].values), frame["y"].values

def train_forecaster(monthly, target, label):
    for lags in (3, 2):
        extras, frame = make_lag_frame(monthly, target, lags)
        if len(frame) >= MIN_FORECAST_ROWS:
            break
    if len(frame) < MIN_FORECAST_ROWS:
        need = MIN_FORECAST_ROWS + 2
        return None, f"only {len(frame)} usable month-windows; need at least {MIN_FORECAST_ROWS} (about {need}+ consecutive months for one person)"
    parts = split_by_time(frame)
    if parts is None:
        return None, "not enough rows to keep a separate train, validation and test set"
    train, val, test = parts
    (xt, yt), (xv, yv), (xe, ye) = stack(train), stack(val), stack(test)
    base_idx = lags * (1 + len(extras))
    models = {"baseline": None, "ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0))}
    if len(train) >= 300:
        models["hgb"] = HistGradientBoostingRegressor(random_state=SEED, max_iter=300, learning_rate=0.05)
    def predict(name, mdl, x):
        return x[:, base_idx] if name == "baseline" else mdl.predict(x)
    val_mae = {}
    for name, mdl in models.items():
        if mdl is not None:
            mdl.fit(xt, yt)
        val_mae[name] = float(np.mean(np.abs(predict(name, mdl, xv) - yv)))
    best = min(val_mae, key=val_mae.get)
    resid = yv - predict(best, models[best], xv)
    q_lo, q_hi = float(np.quantile(resid, 0.10)), float(np.quantile(resid, 0.90))
    pred_e = predict(best, models[best], xe)
    p50, p10, p90 = pred_e * test.scale.values, (pred_e + q_lo) * test.scale.values, (pred_e + q_hi) * test.scale.values
    actual = ye * test.scale.values
    base_pred = xe[:, base_idx] * test.scale.values
    metrics = {
        "target": target, "lags": lags, "chosen_model": best, "validation_mae_ratio": val_mae,
        "test_rows": int(len(test)), "mae": float(np.mean(np.abs(p50 - actual))),
        "baseline_mae": float(np.mean(np.abs(base_pred - actual))),
        "coverage_p10_p90": float(np.mean((actual >= p10) & (actual <= p90))),
        "below_p10": float(np.mean(actual < p10)), "above_p90": float(np.mean(actual > p90)),
        "beats_baseline": bool(np.mean(np.abs(p50 - actual)) < np.mean(np.abs(base_pred - actual))),
    }
    x_all, y_all = np.vstack([xt, xv, xe]), np.concatenate([yt, yv, ye])
    final = models[best]
    if final is not None:
        final.fit(x_all, y_all)
    bundle = {"model": final, "kind": best, "lags": lags, "extras": extras, "target": target,
              "q_lo": q_lo, "q_hi": q_hi, "base_idx": base_idx}
    return (bundle, metrics, test.assign(p10=p10, p50=p50, p90=p90, actual=actual)), None

def predict_next(bundle, history, next_month):
    lags, extras, target = bundle["lags"], bundle["extras"], bundle["target"]
    window = history.sort_values("month").tail(lags)
    if len(window) < lags:
        return {"status": "unknown", "reason": f"need {lags} months of history, got {len(window)}"}
    use_income = window["income"].mean() > 0 if "income" in window else False
    scale = max(float(window["income"].mean()) if use_income else float(window[target].abs().mean()), 1.0)
    feats = [window[target].iloc[-k] / scale for k in range(1, lags + 1)]
    for e in extras:
        feats += [window[e].iloc[-k] / scale for k in range(1, lags + 1)]
    feats += [window[target].mean() / scale, np.sin(2 * np.pi * next_month / 12), np.cos(2 * np.pi * next_month / 12)]
    x = np.array([feats])
    ratio = x[:, bundle["base_idx"]] if bundle["model"] is None else bundle["model"].predict(x)
    return {"status": "ok", "p10": float((ratio[0] + bundle["q_lo"]) * scale), "p50": float(ratio[0] * scale),
            "p90": float((ratio[0] + bundle["q_hi"]) * scale)}

def show_forecast(label, metrics, test_frame):
    print(f"\n== {label} ==")
    print(json.dumps(metrics, indent=2))
    if not metrics["beats_baseline"]:
        print("NOTE: the model does not beat the simple average baseline on the test months. Treat the P50 as the baseline.")
    if metrics["test_rows"] < 30:
        print("CAUTION: fewer than 30 test rows, so these numbers are only indicative.")
    one = test_frame.sort_values("month").head(40)
    plt.figure(figsize=(8, 3.3))
    plt.plot(range(len(one)), one["actual"].values, "k.-", label="actual")
    plt.plot(range(len(one)), one["p50"].values, "b-", label="P50")
    plt.fill_between(range(len(one)), one["p10"].values, one["p90"].values, alpha=0.25, label="P10-P90")
    plt.title(label); plt.legend(); plt.show()

### 5a. Expense forecaster (from transactions, or monthly financials)

In [ ]:
if MONTHLY is None:
    skip("expense_forecaster", "no monthly data")
else:
    out, why = train_forecaster(MONTHLY, "expenses", "expenses")
    if out is None:
        skip("expense_forecaster", why)
    else:
        bundle, metrics, test_frame = out
        BUNDLES["expense_forecaster"] = bundle
        RESULTS["expense_forecaster"] = metrics
        STATUS["expense_forecaster"] = "TRAINED"
        show_forecast("Expense forecast (next month)", metrics, test_frame)

### 5b. Savings-capacity forecaster (next-month surplus)

In [ ]:
if MONTHLY is None:
    skip("savings_forecaster", "no monthly data")
elif float(MONTHLY["income"].sum()) <= 0:
    skip("savings_forecaster", "no income found in the data, so surplus cannot be forecast")
else:
    out, why = train_forecaster(MONTHLY, "surplus", "surplus")
    if out is None:
        skip("savings_forecaster", why)
    else:
        bundle, metrics, test_frame = out
        BUNDLES["savings_forecaster"] = bundle
        RESULTS["savings_forecaster"] = metrics
        STATUS["savings_forecaster"] = "TRAINED"
        show_forecast("Savings capacity: surplus forecast (next month)", metrics, test_frame)

In [ ]:
if "savings_forecaster" in BUNDLES:
    group = MONTHLY.groupby("group")["month"].nunique().idxmax()
    history = MONTHLY[MONTHLY.group == group]
    next_month = ((history["month"].max() + 1).month)
    forecast = predict_next(BUNDLES["savings_forecaster"], history, next_month)
    print(f"Next-month surplus forecast for '{group}':", {k: (round(v) if isinstance(v, float) else v) for k, v in forecast.items()})

## 6. Merchant to category classifier (from transactions)

Learns **merchant text -> category** (character n-grams, so it handles `SWIGGY*1234` style names). Two honest tests are reported:

- **seen merchants:** new rows from merchants the model has already seen (the everyday case, like a user's regular shops).
- **unseen merchants:** whole merchants held out (the hard case). With few merchants per category this number can be low.

In [ ]:
def fit_categorizer(rows):
    pipe = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), lowercase=True, min_df=1),
                         LogisticRegression(max_iter=2000, class_weight="balanced"))
    pipe.fit(rows.merchant, rows.category)
    return pipe

def score_categorizer(pipe, train_rows, test_rows):
    pred = pipe.predict(test_rows.merchant)
    proba = pipe.predict_proba(test_rows.merchant)
    classes = pipe[-1].classes_
    top3 = np.mean([t in classes[np.argsort(-p)[:3]] for t, p in zip(test_rows.category, proba)])
    majority = train_rows.category.value_counts().idxmax()
    return {"train_rows": int(len(train_rows)), "test_rows": int(len(test_rows)),
            "accuracy": float(accuracy_score(test_rows.category, pred)),
            "macro_f1": float(f1_score(test_rows.category, pred, average="macro")),
            "top3_accuracy": float(top3),
            "majority_baseline_accuracy": float(np.mean(test_rows.category == majority))}

def train_categorizer(tidy):
    data = tidy[(tidy.category.str.len() > 0) & (tidy.merchant.str.len() > 0)].copy()
    data["key"] = data.merchant.str.lower().str.replace(r"[^a-z]+", " ", regex=True).str.strip()
    counts = data.category.value_counts()
    data = data[data.category.isin(counts[counts >= 3].index)].reset_index(drop=True)
    if len(data) < 50 or data.category.nunique() < 2:
        return None, f"need at least 50 rows and 2 categories with merchant text; found {len(data)} rows, {data.category.nunique()} categories"
    metrics = {"classes": int(data.category.nunique()), "distinct_merchants": int(data.key.nunique())}
    tr_idx, te_idx = train_test_split(np.arange(len(data)), test_size=0.2, random_state=SEED, stratify=data.category)
    seen_train, seen_test = data.iloc[tr_idx], data.iloc[te_idx]
    metrics["seen_merchants"] = score_categorizer(fit_categorizer(seen_train), seen_train, seen_test)
    if data.key.nunique() >= 10:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
        gtr, gte = next(splitter.split(data, groups=data.key.values))
        unseen_train, unseen_test = data.iloc[gtr], data.iloc[gte]
        if unseen_train.category.nunique() >= 2:
            metrics["unseen_merchants"] = score_categorizer(fit_categorizer(unseen_train), unseen_train, unseen_test)
    return (fit_categorizer(data), metrics), None

if TIDY is None:
    skip("categorizer", "transactions.csv not found")
else:
    out, why = train_categorizer(TIDY)
    if out is None:
        skip("categorizer", why)
    else:
        BUNDLES["categorizer"], RESULTS["categorizer"] = out
        STATUS["categorizer"] = "TRAINED"
        print(json.dumps(RESULTS["categorizer"], indent=2))
        for sample in ["swiggy order 4521", "uber trip", "monthly rent payment"]:
            print(f"  {sample!r:28} -> {BUNDLES['categorizer'].predict([sample])[0]}")

## 7. Risk profiler (from `risk_questionnaire.csv`)

Learns the **risk category** (and the **risk score**) from the questionnaire **answers**.
Answers can be separate columns (`q1, q2, ...`) or one column like `"3,1,4,2"`; both are handled.

In [ ]:
def expand_answers(raw, c):
    answers_col = c.get("answers")
    if answers_col is None or answers_col in (c.get("score"), c.get("risk_category")):
        return None
    if pd.api.types.is_numeric_dtype(raw[answers_col]):
        return None
    series = raw[answers_col].astype(str).str.strip()
    for sep, is_regex in ((",", False), (";", False), ("|", False), (r"\s+", True)):
        parts = series.str.split(sep, regex=is_regex, expand=True)
        if parts.shape[1] >= 2 and bool(parts.notna().all().all()):
            parts.columns = [f"answer_{i + 1}" for i in range(parts.shape[1])]
            numeric = parts.apply(pd.to_numeric, errors="coerce")
            return numeric.where(numeric.notna().all(), parts.apply(lambda s: s.str.strip()))
    return None

def prepare_risk(raw):
    cols = list(raw.columns)
    c = {f: pick("risk_questionnaire", f, cols) for f in ("score", "risk_category", "answers", "user", "date")}
    if c["risk_category"] == c["score"]:
        c["risk_category"] = None
    print("DETECTED risk_questionnaire ->", c)
    score = to_num(raw[c["score"]]) if c["score"] else None
    category = raw[c["risk_category"]].astype(str).str.strip() if c["risk_category"] else None
    category_source = "file"
    if category is None and score is not None:
        category = pd.qcut(score.rank(method="first"), 3, labels=["Low", "Medium", "High"]).astype(str)
        category_source = "derived from score tertiles (the file had no category column)"
    if category is None:
        raise ValueError("risk_questionnaire.csv needs a risk category or risk score column (set COLUMN_OVERRIDE).")
    expanded = expand_answers(raw, c)
    if expanded is not None:
        features = expanded
    else:
        skip_cols = {c["score"], c["risk_category"], c["user"], c["date"]}
        features = raw[[col for col in cols if col not in skip_cols]].copy()
    features = features.loc[:, features.nunique(dropna=True) > 1]
    if features.shape[1] == 0:
        raise ValueError("no usable answer columns found")
    return features, category, score, category_source

def train_risk(raw):
    features, category, score, category_source = prepare_risk(raw)
    numeric = [col for col in features.columns if pd.api.types.is_numeric_dtype(features[col])]
    categorical = [col for col in features.columns if col not in numeric]
    for col in categorical:
        features[col] = features[col].astype(str)
    prep_template = ColumnTransformer([("num", SimpleImputer(strategy="median"), numeric),
                                       ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]), categorical)])
    n = len(features)
    counts = category.value_counts()
    if n < 30 or len(counts) < 2:
        return None, f"need at least 30 rows and 2 risk categories; found {n} rows, {len(counts)} categories"
    stratify = category if counts.min() >= 2 else None
    idx_train, idx_test = train_test_split(np.arange(n), test_size=0.2, random_state=SEED, stratify=stratify)
    clf = Pipeline([("prep", clone(prep_template)), ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED))])
    clf.fit(features.iloc[idx_train], category.iloc[idx_train])
    pred = clf.predict(features.iloc[idx_test])
    truth = category.iloc[idx_test]
    majority = category.iloc[idx_train].value_counts().idxmax()
    metrics = {"rows": int(n), "category_source": category_source, "classes": sorted(counts.index.tolist()),
               "most_common_category": str(counts.idxmax()),
               "accuracy": float(accuracy_score(truth, pred)), "macro_f1": float(f1_score(truth, pred, average="macro")),
               "majority_baseline_accuracy": float(np.mean(truth == majority))}
    if n >= 50 and counts.min() >= 5:
        folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        cv = cross_val_score(clf, features, category, cv=folds, scoring="accuracy")
        metrics["cv_accuracy_mean"], metrics["cv_accuracy_std"] = float(cv.mean()), float(cv.std())
    labels = sorted(counts.index.tolist())
    metrics["confusion_matrix"] = {"labels": labels, "matrix": confusion_matrix(truth, pred, labels=labels).tolist()}
    reg = None
    if score is not None and score.notna().sum() >= 30:
        ok = score.notna().values
        reg = Pipeline([("prep", clone(prep_template)), ("model", RandomForestRegressor(n_estimators=300, random_state=SEED))])
        tr = [i for i in idx_train if ok[i]]
        te = [i for i in idx_test if ok[i]]
        reg.fit(features.iloc[tr], score.iloc[tr])
        sp = reg.predict(features.iloc[te])
        metrics["score_mae"] = float(mean_absolute_error(score.iloc[te], sp))
        metrics["score_r2"] = float(r2_score(score.iloc[te], sp))
        metrics["score_baseline_mae"] = float(np.mean(np.abs(score.iloc[te] - score.iloc[tr].mean())))
        reg.fit(features[ok], score[ok])
    clf.fit(features, category)
    return ({"classifier": clf, "regressor": reg, "columns": list(features.columns)}, metrics), None

if "risk_questionnaire" not in RAW:
    skip("risk_profiler", "risk_questionnaire.csv not found")
else:
    try:
        out, why = train_risk(RAW["risk_questionnaire"])
    except ValueError as error:
        out, why = None, str(error)
    if out is None:
        skip("risk_profiler", why)
    else:
        BUNDLES["risk_profiler"], RESULTS["risk_profiler"] = out
        STATUS["risk_profiler"] = "TRAINED"
        print(json.dumps(RESULTS["risk_profiler"], indent=2))
        if RESULTS["risk_profiler"]["accuracy"] <= RESULTS["risk_profiler"]["majority_baseline_accuracy"]:
            print("NOTE: accuracy is not above the majority-class baseline; the answers may not carry enough signal or there is too little data.")

## 8. Goal planner (from `financial_goals.csv`)

The goals file has **no label to learn from**, so this is a transparent calculator instead of a trained model:

- `required monthly saving = (target - current savings) / months left`
- Compared with the forecast monthly surplus (if available): **on track** if it fits the cautious (P10) case, **possible** if it fits the typical (P50) case, otherwise **off track**.

In [ ]:
def surplus_bounds(group=None):
    if "savings_forecaster" not in BUNDLES or MONTHLY is None:
        return None
    results = {}
    for g, history in MONTHLY.groupby("group"):
        nm = (history["month"].max() + 1).month
        res = predict_next(BUNDLES["savings_forecaster"], history, nm)
        if res["status"] == "ok":
            results[g] = res
    if not results:
        return None
    if group is not None and group in results:
        return results[group]
    if len(results) == 1:
        return next(iter(results.values()))
    return {k: float(np.median([r[k] for r in results.values()])) for k in ("p10", "p50", "p90")}

def plan_goals(raw):
    cols = list(raw.columns)
    c = {f: pick("financial_goals", f, cols) for f in ("goal", "target", "current", "deadline", "user")}
    print("DETECTED financial_goals ->", c)
    if c["target"] is None or c["deadline"] is None:
        raise ValueError("financial_goals.csv needs target amount and deadline columns (set COLUMN_OVERRIDE).")
    as_of = pd.Timestamp(AS_OF_DATE) if AS_OF_DATE else pd.Timestamp.today().normalize()
    plan = pd.DataFrame({"user": raw[c["user"]].astype(str) if c["user"] else "user_0",
                         "goal": raw[c["goal"]].astype(str) if c["goal"] else [f"goal_{i + 1}" for i in range(len(raw))],
                         "target": to_num(raw[c["target"]]),
                         "current": to_num(raw[c["current"]]).fillna(0) if c["current"] else 0.0,
                         "deadline": to_date(raw[c["deadline"]])}).dropna(subset=["target", "deadline"])
    plan["remaining"] = (plan.target - plan.current).clip(lower=0)
    plan["progress_pct"] = np.where(plan.target > 0, 100 * plan.current / plan.target, 0).round(1)
    plan["months_left"] = ((plan.deadline.dt.year - as_of.year) * 12 + plan.deadline.dt.month - as_of.month).clip(lower=0)
    plan["required_monthly"] = np.where(plan.months_left > 0, plan.remaining / plan.months_left.where(plan.months_left > 0, 1), np.nan)
    statuses = []
    for _, row in plan.iterrows():
        bounds = surplus_bounds(row["user"] if row["user"] != "user_0" else None)
        if row.remaining <= 0:
            statuses.append("achieved")
        elif row.months_left == 0:
            statuses.append("deadline passed")
        elif bounds is None:
            statuses.append("needs surplus forecast")
        elif row.required_monthly <= bounds["p10"]:
            statuses.append("on track (even in the cautious case)")
        elif row.required_monthly <= bounds["p50"]:
            statuses.append("possible (typical case only)")
        else:
            statuses.append("off track")
    plan["status"] = statuses
    return plan, as_of

if "financial_goals" not in RAW:
    skip("goal_planner", "financial_goals.csv not found")
else:
    try:
        GOAL_PLAN, as_of = plan_goals(RAW["financial_goals"])
        STATUS["goal_planner"] = "READY (rules, not trained)"
        print("as of:", as_of.date())
        display(GOAL_PLAN.round(1))
        total = GOAL_PLAN.loc[GOAL_PLAN.required_monthly.notna(), "required_monthly"].sum()
        bounds = surplus_bounds()
        print(f"Total required saving per month across goals: {money(total)}")
        if bounds:
            print(f"Forecast surplus next month: cautious {money(bounds['p10'])} | typical {money(bounds['p50'])} | good {money(bounds['p90'])}")
        GOAL_PLAN.to_csv(f"{OUT_DIR}/goal_plan.csv", index=False)
        RESULTS["goal_planner"] = {"goals": int(len(GOAL_PLAN)), "status_counts": GOAL_PLAN.status.value_counts().to_dict()}
    except ValueError as error:
        skip("goal_planner", str(error))

## 9. Investment catalogue (cleaning) and recommendation feedback (acceptance model)

**Catalogue:** text like `"3 years"`, `"Low"`, `"1.5%"` is converted into numbers: `risk_level` (1 low ... 4 very high), `liquidity_score` (1 locked ... 3 instant), `fee_pct`, `lock_in_months`. A bare number in the lock-in column is treated as **months**.

**Feedback:** learns whether the user will **accept** a suggested option. The free-text `reason` is **not** used as an input (it is only known *after* the decision). It is shown as an analysis table instead.

In [ ]:
def risk_level_of(value):
    text = str(value).lower()
    if "very high" in text: return 4
    if "very low" in text or text.strip() in ("low", "1"): return 1
    if "high" in text: return 3
    if "moderate" in text or "medium" in text or "mid" in text: return 2
    if "low" in text: return 1
    number = pd.to_numeric(text, errors="coerce")
    return int(min(max(round(number), 1), 4)) if pd.notna(number) else np.nan

def liquidity_of(value):
    text = str(value).lower()
    if any(k in text for k in ("instant", "daily", "high", "same day", "t+1")): return 3
    if any(k in text for k in ("medium", "moderate", "week", "days")): return 2
    if any(k in text for k in ("low", "locked", "illiquid", "none")): return 1
    number = pd.to_numeric(text, errors="coerce")
    return int(min(max(round(number), 1), 3)) if pd.notna(number) else np.nan

def fee_of(value):
    match = re.search(r"\d+(?:\.\d+)?", str(value))
    return float(match.group()) if match else 0.0

def lock_in_months_of(value):
    text = str(value).lower().strip()
    if text in ("", "nan", "none", "no", "nil", "na", "n/a", "-", "liquid", "no lock-in", "no lock in"):
        return 0.0
    match = re.search(r"(\d+(?:\.\d+)?)\s*(year|yr|y\b)", text)
    if match: return float(match.group(1)) * 12
    match = re.search(r"(\d+(?:\.\d+)?)\s*(month|mo\b|m\b)", text)
    if match: return float(match.group(1))
    match = re.search(r"\d+(?:\.\d+)?", text)
    return float(match.group()) if match else 0.0

def name_key(value):
    return norm_name(value)

def clean_catalogue(raw):
    cols = list(raw.columns)
    c = {f: pick("investment_catalogue", f, cols) for f in ("inv_type", "risk", "liquidity", "fees", "lock_in", "eligibility")}
    print("DETECTED investment_catalogue ->", c)
    if c["inv_type"] is None:
        raise ValueError("investment_catalogue.csv needs an investment type/name column (set COLUMN_OVERRIDE).")
    cat = pd.DataFrame({"name": raw[c["inv_type"]].astype(str).str.strip()})
    cat["key"] = cat["name"].map(name_key)
    cat["risk_level"] = raw[c["risk"]].map(risk_level_of) if c["risk"] else np.nan
    cat["liquidity_score"] = raw[c["liquidity"]].map(liquidity_of) if c["liquidity"] else np.nan
    cat["fee_pct"] = raw[c["fees"]].map(fee_of) if c["fees"] else 0.0
    cat["lock_in_months"] = raw[c["lock_in"]].map(lock_in_months_of) if c["lock_in"] else 0.0
    cat["eligibility"] = raw[c["eligibility"]].astype(str) if c["eligibility"] else ""
    cat["risk_level"] = cat["risk_level"].fillna(cat["risk_level"].median() if cat["risk_level"].notna().any() else 2)
    cat["liquidity_score"] = cat["liquidity_score"].fillna(cat["liquidity_score"].median() if cat["liquidity_score"].notna().any() else 2)
    return cat.drop_duplicates("key").reset_index(drop=True)

CATALOGUE = None
if "investment_catalogue" in RAW:
    try:
        CATALOGUE = clean_catalogue(RAW["investment_catalogue"])
        STATUS["investment_catalogue"] = "CLEANED"
        display(CATALOGUE)
        CATALOGUE.to_csv(f"{OUT_DIR}/catalogue_clean.csv", index=False)
    except ValueError as error:
        skip("investment_catalogue", str(error))
else:
    skip("investment_catalogue", "investment_catalogue.csv not found")

In [ ]:
def parse_label(series):
    text = series.astype(str).str.lower().str.strip()
    positive = text.str.contains(r"accept|yes|true|approved|positive|liked|taken|^1(\.0)?$", regex=True)
    negative = text.str.contains(r"reject|no\b|false|declin|negative|dismiss|ignored|^0(\.0)?$", regex=True)
    label = pd.Series(np.nan, index=series.index)
    label[positive & ~negative] = 1.0
    label[negative & ~positive] = 0.0
    return label

def prepare_feedback(raw):
    cols = list(raw.columns)
    c = {f: pick("recommendation_feedback", f, cols) for f in ("option", "label", "reason", "user", "date")}
    print("DETECTED recommendation_feedback ->", c)
    if c["option"] is None or c["label"] is None:
        raise ValueError("recommendation_feedback.csv needs a suggested-option column and an accepted/rejected column (set COLUMN_OVERRIDE).")
    label = parse_label(raw[c["label"]])
    if "reject" in norm_name(c["label"]) and "accept" not in norm_name(c["label"]):
        label = 1.0 - label
    frame = raw.copy()
    frame["_label"] = label
    frame = frame[frame["_label"].notna()].reset_index(drop=True)
    frame["option_key"] = frame[c["option"]].astype(str).map(name_key)
    drop = {c["label"], c["reason"], c["user"], c["date"], "_label", "option_key"}
    features = frame[[col for col in frame.columns if col not in drop]].copy()
    features = features.rename(columns={c["option"]: "option"})
    if CATALOGUE is not None:
        join = CATALOGUE.set_index("key")[["risk_level", "liquidity_score", "fee_pct", "lock_in_months"]]
        extra = join.reindex(frame["option_key"]).reset_index(drop=True)
        extra.columns = [f"opt_{col}" for col in extra.columns]
        features = pd.concat([features, extra], axis=1)
    features = features.loc[:, features.nunique(dropna=True) > 1]
    return features, frame["_label"].astype(int), frame, c

def train_acceptance(raw):
    features, y, frame, c = prepare_feedback(raw)
    n, pos = len(y), int(y.sum())
    if n < 50 or pos < 5 or n - pos < 5:
        return None, f"need at least 50 labelled rows with at least 5 accepted and 5 rejected; found {n} rows ({pos} accepted)"
    numeric = [col for col in features.columns if pd.api.types.is_numeric_dtype(features[col])]
    categorical = [col for col in features.columns if col not in numeric]
    for col in categorical:
        features[col] = features[col].astype(str)
    def build(model):
        prep = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), numeric),
                                  ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]), categorical)])
        return Pipeline([("prep", prep), ("model", model)])
    candidates = {"logistic": build(LogisticRegression(max_iter=3000, class_weight="balanced")),
                  "forest": build(RandomForestClassifier(n_estimators=300, class_weight="balanced", min_samples_leaf=2, random_state=SEED))}
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_auc = {k: float(cross_val_score(m, features, y, cv=folds, scoring="roc_auc").mean()) for k, m in candidates.items()}
    best = max(cv_auc, key=cv_auc.get)
    idx_train, idx_test = train_test_split(np.arange(n), test_size=0.2, random_state=SEED, stratify=y)
    model = candidates[best]
    model.fit(features.iloc[idx_train], y.iloc[idx_train])
    proba = model.predict_proba(features.iloc[idx_test])[:, 1]
    metrics = {"rows": int(n), "accepted_rate": float(y.mean()), "cv_auc": cv_auc, "chosen_model": best,
               "test_auc": float(roc_auc_score(y.iloc[idx_test], proba)),
               "test_accuracy": float(accuracy_score(y.iloc[idx_test], (proba >= 0.5).astype(int))),
               "majority_baseline_accuracy": float(max(y.iloc[idx_test].mean(), 1 - y.iloc[idx_test].mean()))}
    model.fit(features, y)
    defaults = {col: (features[col].median() if col in numeric else features[col].mode().iloc[0]) for col in features.columns}
    reasons = None
    if c["reason"]:
        rejected = frame[frame["_label"] == 0][c["reason"]].astype(str).str.strip().str.lower()
        reasons = rejected.value_counts().head(10).to_dict()
    return ({"model": model, "columns": list(features.columns), "defaults": defaults, "reasons": reasons}, metrics), None

if "recommendation_feedback" not in RAW:
    skip("acceptance_model", "recommendation_feedback.csv not found")
else:
    try:
        out, why = train_acceptance(RAW["recommendation_feedback"])
    except ValueError as error:
        out, why = None, str(error)
    if out is None:
        skip("acceptance_model", why)
    else:
        BUNDLES["acceptance_model"], RESULTS["acceptance_model"] = out
        STATUS["acceptance_model"] = "TRAINED"
        print(json.dumps(RESULTS["acceptance_model"], indent=2))
        if BUNDLES["acceptance_model"]["reasons"]:
            print("\nTop rejection reasons:", BUNDLES["acceptance_model"]["reasons"])

## 10. Recommendation engine (catalogue + risk profile + goals + acceptance model)

1. **Eligibility filter:** risk not above the user's risk level, lock-in not longer than the goal horizon, liquidity at least what is needed.
2. **Score:** acceptance probability (if the feedback model exists) + low fees. The `eligibility` text from the file is shown but **cannot be checked automatically**, so please verify it manually.

This produces suggestions for exploring options, not financial advice.

In [ ]:
RISK_LEVEL_OF_CATEGORY = lambda text: 1 if "low" in str(text).lower() else 3 if "high" in str(text).lower() else 2

def recommend(profile, top_k=5):
    if CATALOGUE is None:
        return None
    options = CATALOGUE.copy()
    options = options[(options.risk_level <= profile["risk_level"]) &
                      (options.lock_in_months <= profile["horizon_months"]) &
                      (options.liquidity_score >= profile.get("min_liquidity", 1))]
    if options.empty:
        return pd.DataFrame(columns=["name", "score", "risk_level", "liquidity_score", "fee_pct", "lock_in_months", "eligibility"])
    fee_part = 1 - options.fee_pct / max(float(CATALOGUE.fee_pct.max()), 1e-9)
    if "acceptance_model" in BUNDLES:
        bundle = BUNDLES["acceptance_model"]
        rows = pd.DataFrame([bundle["defaults"]] * len(options))
        if "option" in rows:
            rows["option"] = options["name"].values
        for col, src in (("opt_risk_level", "risk_level"), ("opt_liquidity_score", "liquidity_score"),
                         ("opt_fee_pct", "fee_pct"), ("opt_lock_in_months", "lock_in_months")):
            if col in rows:
                rows[col] = options[src].values
        probability = bundle["model"].predict_proba(rows[bundle["columns"]])[:, 1]
        options = options.assign(acceptance_probability=probability, score=0.7 * probability + 0.3 * fee_part.values)
    else:
        options = options.assign(score=fee_part.values)
    return options.sort_values("score", ascending=False).head(top_k)[
        [c for c in ("name", "score", "acceptance_probability", "risk_level", "liquidity_score", "fee_pct", "lock_in_months", "eligibility") if c in options]]

if CATALOGUE is None:
    skip("recommender", "investment catalogue not available")
else:
    level = 2
    if "risk_profiler" in RESULTS:
        level = RISK_LEVEL_OF_CATEGORY(RESULTS["risk_profiler"]["most_common_category"])
    horizon = 36
    if "GOAL_PLAN" in globals() and len(GOAL_PLAN):
        horizon = int(max(GOAL_PLAN.months_left.median(), 1))
    demo_profile = {"risk_level": level, "horizon_months": horizon, "min_liquidity": 1}
    STATUS["recommender"] = "READY (rules" + (" + acceptance model)" if "acceptance_model" in BUNDLES else ")")
    print("Demo profile:", demo_profile)
    display(recommend(demo_profile).round(3))

## 11. Save everything

Saved in `outputs_tabular/`: trained models (`.joblib`), `metrics.json`, `status.json`, cleaned catalogue, goal plan, and a SHA-256 manifest.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

for name, bundle in BUNDLES.items():
    joblib.dump(bundle, f"{OUT_DIR}/models/{name}.joblib")
json.dump(RESULTS, open(f"{OUT_DIR}/metrics.json", "w"), indent=2, default=str)
json.dump(STATUS, open(f"{OUT_DIR}/status.json", "w"), indent=2)
manifest = {str(p.relative_to(OUT_DIR)).replace(os.sep, "/"): {"sha256": sha256(p), "bytes": p.stat().st_size}
            for p in sorted(Path(OUT_DIR).rglob("*")) if p.is_file() and p.name != "manifest.json"}
json.dump(manifest, open(f"{OUT_DIR}/manifest.json", "w"), indent=2)

print("\n==== SUMMARY ====")
for key in ["expense_forecaster", "savings_forecaster", "categorizer", "risk_profiler", "goal_planner",
            "investment_catalogue", "acceptance_model", "recommender"]:
    print(f"{key:22s} {STATUS.get(key, 'NOT RUN')}")
print("\nfiles:", sorted(manifest))

## Notes
- Models are saved with `joblib` (scikit-learn). For the Android app they would need converting (for example to ONNX or a small PyTorch network for ExecuTorch); tell me which models you want on the device.
- If a section says SKIPPED, the message explains exactly what is missing (more months, more rows, or a column the notebook could not find).
- To keep the run reproducible, the seed is fixed and library versions are saved in `metrics.json`.